# 03 — Entraînement & comparaison
Entraîne les 4 modèles + baseline LLM, avec tracking MLflow.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # racine ml/ -> import src.*
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.features.pipeline import load_dataset, build_feature_matrix, get_target
from sklearn.model_selection import train_test_split
df = load_dataset(); X,_ = build_feature_matrix(df); y = get_target(df)
print('X:', X.shape, '| conversion:', round(y.mean(),3))

## Split stratifié 80/20

In [ ]:
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print('train', Xtr.shape, 'test', Xte.shape)
print('conversion train=%.3f test=%.3f' % (ytr.mean(), yte.mean()))

## Entraînement complet + MLflow
On lance le pipeline complet (`src/models/train.py`) qui logge dans MLflow, sauvegarde le meilleur modèle et met à jour le registre.

In [ ]:
from src.models.train import main as train_main
train_main()

## Tableau comparatif final

In [ ]:
import json
m = json.load(open('../artifacts/metrics.json', encoding='utf-8'))
pd.DataFrame(m['models'])[['model_name','auc_roc','f1_score','precision_at_20','lift_at_20']]
  .sort_values('auc_roc', ascending=False)

## Courbe ROC (4 modèles + LLM)

In [ ]:
from IPython.display import Image
Image('../artifacts/roc_curves.png')

## Meilleur modèle sauvegardé

In [ ]:
print('best_model:', m['best_model'])